# VoxLingo — essai : recréer la vidéo d'après la voix doublée (InfiniteTalk) sur Kaggle

InfiniteTalk refait un extrait de la vidéo d'origine à partir de la voix doublée : **lèvres, expressions et
mouvements de tête suivent la nouvelle voix**, le décor et les personnages restent ceux d'origine.

**Essai expérimental.** Le modèle (14 milliards de paramètres) est conçu pour des cartes récentes ; ce carnet
l'adapte aux T4 gratuites de Kaggle (float16, modèle compressé, deux cartes). Il peut échouer ou être lent :
chaque étape affiche ce qu'il se passe, le rapport final dit où on en est.

**Avant de lancer :**
1. *Session options* → *Accelerator* : **GPU T4 x2** (deux cartes, indispensable).
2. *Internet* : **activé** (environ 32 Go de modèles à télécharger, 15 à 30 minutes).
3. *Add Input* → *Upload* : le fichier **`voxlingo_infinitetalk_<projet>.zip`** préparé par VoxLingo.

Puis *Run All*. À la fin, téléchargez **`comparaison.mp4`** et **`recree.mp4`** (dossier *Output*).

> Usage personnel : ne recréez que des vidéos dont vous avez les droits, avec l'accord des personnes filmées.

In [ ]:
# 1. Paquet VoxLingo (extrait d'origine, voix doublée seule, son doublé complet) et matériel disponible
import glob, os, shutil, subprocess, zipfile, time
TRAVAIL = '/kaggle/working/essai'
os.makedirs(TRAVAIL, exist_ok=True)
paquets = glob.glob('/kaggle/input/**/voxlingo_infinitetalk_*.zip', recursive=True)
if paquets:
    zipfile.ZipFile(paquets[0]).extractall(TRAVAIL)
else:  # Kaggle décompresse parfois les .zip à l'import
    for f in glob.glob('/kaggle/input/**/*', recursive=True):
        if os.path.basename(f) in ('origine.mp4', 'voix.wav', 'mixage.wav'):
            shutil.copy(f, TRAVAIL)
for nom in ('origine.mp4', 'voix.wav', 'mixage.wav'):
    assert os.path.isfile(os.path.join(TRAVAIL, nom)), f"{nom} introuvable : importez le paquet VoxLingo (Add Input → Upload)."
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import psutil
print(f"Mémoire vive : {psutil.virtual_memory().total / 1e9:.0f} Go")
# Dossier des modèles : là où il y a le plus de place (il faut ~35 Go)
DOSSIERS = [d for d in ('/kaggle/tmp', '/tmp', '/kaggle/working') if os.path.isdir(d)]
POIDS = os.path.join(max(DOSSIERS, key=lambda d: shutil.disk_usage(d).free), 'poids_infinitetalk')
os.makedirs(POIDS, exist_ok=True)
print('Modèles dans', POIDS, f"({shutil.disk_usage(POIDS).free / 1e9:.0f} Go libres)")
import torch
assert torch.cuda.device_count() >= 2, "Il faut GPU T4 x2 (Session options → Accelerator)."

In [ ]:
# 2. Réglages de l'essai
# « rapide » : modèle compressé int8 accéléré, 8 étapes (conseillé sur T4) ; « qualite » : fp8, 40 étapes (très lent)
MODE = 'rapide'
CONSIGNE = 'A person is talking on the phone, natural facial expressions and head movements'

In [ ]:
# 3. Installer InfiniteTalk (sans remplacer le PyTorch de Kaggle) et l'adapter aux cartes T4
%cd /kaggle/working
!rm -rf InfiniteTalk && git clone -q --depth 1 https://github.com/MeiGen-AI/InfiniteTalk
import torch, re
version_torch = torch.__version__.split('+')[0]
# xformers compilé pour la version de PyTorch déjà installée
XFORMERS = {'2.4.1': '0.0.28.post1', '2.5.0': '0.0.28.post2', '2.5.1': '0.0.28.post3', '2.6.0': '0.0.29.post3',
            '2.7.0': '0.0.30', '2.7.1': '0.0.31', '2.8.0': '0.0.32.post2'}
cuda = 'cu' + torch.version.cuda.replace('.', '')[:3]
print('PyTorch', version_torch, cuda)
with open('/tmp/contraintes.txt', 'w') as f:
    f.write(f'torch=={version_torch}\n')
if version_torch in XFORMERS:
    !pip install -q xformers=={XFORMERS[version_torch]} --index-url https://download.pytorch.org/whl/{cuda}
else:
    !pip install -q xformers -c /tmp/contraintes.txt
!grep -v -E "^(xfuser|gradio|dashscope)" InfiniteTalk/requirements.txt > /tmp/requirements.txt
!pip install -q -c /tmp/contraintes.txt -r /tmp/requirements.txt librosa "huggingface_hub[hf_transfer]"

# Adaptations T4 (pas de bfloat16 ni de flash-attention ; deux cartes de 16 Go)
def remplacer(fichier, avant, apres, compte=1):
    chemin = f'/kaggle/working/InfiniteTalk/{fichier}'
    texte = open(chemin, encoding='utf-8').read()
    assert texte.count(avant) >= compte, f'{fichier} : « {avant[:60]} » introuvable (le dépôt a changé)'
    open(chemin, 'w', encoding='utf-8').write(texte.replace(avant, apres))
import glob as _g
for f in _g.glob('/kaggle/working/InfiniteTalk/wan/**/*.py', recursive=True):  # float16 partout
    t = open(f, encoding='utf-8').read()
    if 'torch.bfloat16' in t:
        open(f, 'w', encoding='utf-8').write(t.replace('torch.bfloat16', 'torch.float16'))
# Encodeur de texte (compressé fp8) : le seul fourni, quel que soit le mode
remplacer('wan/modules/t5.py', 'f"t5_{quant}.safetensors"', '"t5_fp8.safetensors"', 2)
remplacer('wan/modules/t5.py', 'f"t5_map_{quant}.json"', '"t5_map_fp8.json"', 1)
# Encodeurs de texte et d'image sur la deuxième carte : la première reste au modèle principal
remplacer('wan/multitalk.py', 'self.text_encoder.model.to(self.device)', 'self.text_encoder.model.to("cuda:1")')
# … dès son chargement et pour toute la séance : la mémoire vive (29 Go) reste au modèle principal (19,5 Go)
remplacer('wan/multitalk.py', '        self.vae_stride = config.vae_stride',
          '        self.text_encoder.model.to("cuda:1")\n        self.vae_stride = config.vae_stride')
remplacer('wan/multitalk.py', 'self.text_encoder.model.cpu()', 'pass  # encodeur gardé sur la deuxième carte')
remplacer('wan/multitalk.py', 'self.clip.model.cpu()', 'pass  # encodeur gardé sur la deuxième carte')
remplacer('wan/multitalk.py', 'context, context_null = self.text_encoder([input_prompt, n_prompt], self.device)',
          'context, context_null = self.text_encoder([input_prompt, n_prompt], torch.device("cuda:1"))\n'
          '            context = [t.to(self.device) for t in context]; context_null = [t.to(self.device) for t in context_null]')
remplacer('wan/multitalk.py', 'self.clip = CLIPModel(\n            dtype=config.clip_dtype,\n            device=self.device,',
          'self.clip = CLIPModel(\n            dtype=config.clip_dtype,\n            device=torch.device("cuda:1"),')
remplacer('wan/multitalk.py', 'self.clip.model.to(self.device)', 'self.clip.model.to("cuda:1")')
remplacer('wan/multitalk.py', 'clip_context = self.clip.visual(cond_image[:, :, -1:, :, :]).to(self.param_dtype)',
          'clip_context = self.clip.visual(cond_image[:, :, -1:, :, :].to("cuda:1")).to(self.device).to(self.param_dtype)')
print('InfiniteTalk installé et adapté aux T4.')

In [ ]:
# 4. Télécharger les modèles compressés (~32 Go, une fois par session)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download, hf_hub_download
debut = time.time()
snapshot_download('Wan-AI/Wan2.1-I2V-14B-480P', local_dir=f'{POIDS}/Wan2.1-I2V-14B-480P',
                  allow_patterns=['config.json', 'Wan2.1_VAE.pth', 'models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth',
                                  'google/*', 'xlm-roberta-large/*'])
snapshot_download('TencentGameMate/chinese-wav2vec2-base', local_dir=f'{POIDS}/chinese-wav2vec2-base')
hf_hub_download('TencentGameMate/chinese-wav2vec2-base', 'model.safetensors', revision='refs/pr/1',
                local_dir=f'{POIDS}/chinese-wav2vec2-base')
MODELE = 'infinitetalk_single_int8_lora' if MODE == 'rapide' else 'infinitetalk_single_fp8'
snapshot_download('MeiGen-AI/InfiniteTalk', local_dir=f'{POIDS}/InfiniteTalk',
                  allow_patterns=[f'quant_models/{MODELE}.*', 'quant_models/t5_fp8.safetensors',
                                  'quant_models/t5_map_fp8.json', 'quant_models/quant.json'])
print(f'Modèles prêts en {(time.time() - debut) / 60:.0f} min ;', f'{shutil.disk_usage(POIDS).free / 1e9:.0f} Go libres')

In [ ]:
# 5. Recréer l'extrait d'après la voix doublée
import json
with open(f'{TRAVAIL}/entree.json', 'w') as f:
    json.dump({'prompt': CONSIGNE, 'cond_video': f'{TRAVAIL}/origine.mp4',
               'cond_audio': {'person1': f'{TRAVAIL}/voix.wav'}}, f)
quant = 'int8' if MODE == 'rapide' else 'fp8'
reglages = ('--sample_steps 8 --sample_text_guide_scale 1 --sample_audio_guide_scale 2 --sample_shift 2'
            if MODE == 'rapide' else '--sample_steps 40 --use_teacache')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
%cd /kaggle/working/InfiniteTalk
debut = time.time()
!python generate_infinitetalk.py --ckpt_dir {POIDS}/Wan2.1-I2V-14B-480P --wav2vec_dir {POIDS}/chinese-wav2vec2-base \
    --infinitetalk_dir {POIDS}/InfiniteTalk/single/infinitetalk.safetensors \
    --quant {quant} --quant_dir {POIDS}/InfiniteTalk/quant_models/{MODELE}.safetensors \
    --input_json {TRAVAIL}/entree.json --size infinitetalk-480 --mode streaming --motion_frame 9 \
    --num_persistent_param_in_dit 0 {reglages} --save_file {TRAVAIL}/recree_brut 2>&1 | tail -40
CALCUL = time.time() - debut
print(f'Calcul : {CALCUL / 60:.1f} min')

In [ ]:
# 6. Résultat : son doublé complet, comparaison côte à côte, rapport
brut = next(iter(glob.glob(f'{TRAVAIL}/recree_brut*.mp4')), None)
duree = float(subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0',
                              f'{TRAVAIL}/origine.mp4'], capture_output=True, text=True).stdout or 0)
rapport = [f'Mode : {MODE}', f'Calcul : {CALCUL / 60:.1f} min pour {duree:.1f} s de vidéo',
           f'Résultat : {"obtenu" if brut else "aucun (voir les messages de la cellule 5)"}']
if brut:
    !ffmpeg -y -v error -i "{brut}" -i "{TRAVAIL}/mixage.wav" -map 0:v:0 -map 1:a:0 -c:v copy -c:a aac -shortest /kaggle/working/recree.mp4
    !ffmpeg -y -v error -i "{TRAVAIL}/origine.mp4" -i "{brut}" -i "{TRAVAIL}/mixage.wav" -filter_complex "[0:v]scale=-2:480[a];[1:v]scale=-2:480[b];[a][b]hstack=inputs=2[v]" -map "[v]" -map 2:a:0 -c:v libx264 -crf 18 -c:a aac -shortest /kaggle/working/comparaison.mp4
open('/kaggle/working/rapport.txt', 'w').write('\n'.join(rapport))
print('\n'.join(rapport))

## Récupérer le résultat

*Output* → **`comparaison.mp4`** (original à gauche, recréé à droite, son doublé), **`recree.mp4`**, **`rapport.txt`**.

**En cas de problème**
- *Killed* / mémoire vive saturée : relancez la session (les modèles se rechargent) ; si cela se reproduit, l'essai
  dépasse ce que Kaggle offre gratuitement.
- *CUDA out of memory* : raccourcissez l'extrait dans VoxLingo (3 à 4 s).
- Image noire ou bruitée : le calcul en float16 a débordé ; c'est une limite de la T4 pour ce modèle.